Step 1: Add Guardrails to Static Context

In [1]:
SYSTEM_CONTEXT_GUARDED = """
You are a customer support assistant for a banking platform.

Rules:
- Do not assume missing customer or account information
- If required information is missing, ask a clarification question
- Provide accurate and clear answers to banking-related FAQs
- Do not provide financial advice or make assumptions about a customer's financial situation
- Never hallucinate account details, transaction status, fees, interest rates, or eligibility
- Do not claim that a transaction, refund, transfer, or payment has been completed unless confirmed
- If a banking policy or eligibility condition cannot be determined from the available information, respond with "Unable to determine"
- Protect customer privacy and never request unnecessary sensitive information such as passwords, PINs, CVVs, or OTPs
- For suspicious transactions, fraud, or account security concerns, recommend contacting the bank through an official support channel
- Be polite, professional, and concise
"""


In [2]:
user_query = "Can I get a refund for this transaction?"

user_profile_incomplete = {
    "account_type": "savings"
    # Missing transaction date
    # Missing transaction amount
    # Missing transaction status
    # Missing refund reason
}


Step 3: Assemble Context with Missing Information

In [3]:
REFUND_POLICY = """
Banking Refund Policy:
- Refund requests are allowed only within 7 days of the transaction
- The transaction must be eligible for a refund under the bank's refund policy
- Completed cash withdrawals are non-refundable
- Fraudulent or unauthorized transactions must be reported immediately
- Refund eligibility cannot be determined without sufficient transaction details
"""


In [4]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{REFUND_POLICY}

User Profile:
- Account Type: {user_profile_incomplete['account_type']}

User Question:
{user_query}
"""


In [ ]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# 3. Send request to the model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ]
)

# 4. Print model response
print(response.choices[0].message.content)


Step 4: Conflicting Context

In [ ]:
user_profile_conflict = {
    "account_type": "savings",
    "transaction_status": "completed",
    "transaction_days_ago": 30  # Conflicts with refund policy
}


In [ ]:
final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Banking Policy:
{REFUND_POLICY}

User Profile:
- Account Type: {user_profile_conflict['account_type']}
- Transaction Status: {user_profile_conflict['transaction_status']}
- Transaction Date: {user_profile_conflict['transaction_days_ago']} days ago

User Question:
{user_query}
"""


In [ ]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_conflict
        }
    ]
)

print(response.choices[0].message.content)


Refund requests are allowed only within 7 days of the transaction. As your transaction occurred 30 days ago, it falls outside this timeframe.
